# Smoke tests de OP-10 `write_flows`

Este notebook contiene smoke tests integrados de la operación pública `write_flows()`.

Objetivo:

- verificar que la función pública pueda ejecutarse correctamente en escenarios simples y representativos;
- revisar que materialice bundles `.golondrina` válidos;
- comprobar el comportamiento observable mínimo de `summary`, `parameters`, sidecar y metadata;
- cubrir tanto el caso base como la persistencia opcional de `flow_to_trips`.

Alcance:

- se prueban únicamente smoke tests de **OP-10 `write_flows`**;
- no se incluyen tests helper-level;
- no se incluyen tests de `read_flows`, que corresponden a OP-11;
- los artefactos de persistencia se crean en una carpeta local junto al notebook para poder inspeccionarlos manualmente.

## Bloque 1. Setup visible de smoke tests

Qué prepara:

- una carpeta local `./tmp_smoke_write_flows` para persistir artefactos visibles junto al notebook;
- factories pequeñas para construir `FlowDataset`;
- helpers de lectura de sidecar y salida visual mínima;
- imports necesarios para ejecutar smoke tests de la función pública `write_flows`.

In [1]:
from pathlib import Path
import json
import shutil
import copy

import pandas as pd

from pylondrina.datasets import FlowDataset
from pylondrina.io.flows import (
    write_flows,
    WriteFlowsOptions,
)


SMOKE_ROOT = Path("./tmp_smoke_write_flows")


def show_ok(label: str):
    print(f"OK - {label}")


def reset_smoke_root() -> Path:
    if SMOKE_ROOT.exists():
        shutil.rmtree(SMOKE_ROOT)
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    return SMOKE_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = SMOKE_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def make_flow_df(n_repeat: int = 1) -> pd.DataFrame:
    base = pd.DataFrame(
        {
            "flow_id": ["f_0001", "f_0002", "f_0003"],
            "origin_h3_index": [
                "8828308281fffff",
                "8828308281fffff",
                "8828308285fffff",
            ],
            "destination_h3_index": [
                "8828308287fffff",
                "8828308289fffff",
                "8828308287fffff",
            ],
            "flow_count": [10, 6, 4],
            "flow_value": [10.0, 6.0, 4.0],
            "mode": ["bus", "metro", "bus"],
            "window_start_utc": pd.to_datetime(
                [
                    "2026-01-01T08:00:00Z",
                    "2026-01-01T08:00:00Z",
                    "2026-01-01T09:00:00Z",
                ],
                utc=True,
            ),
            "window_end_utc": pd.to_datetime(
                [
                    "2026-01-01T09:00:00Z",
                    "2026-01-01T09:00:00Z",
                    "2026-01-01T10:00:00Z",
                ],
                utc=True,
            ),
        }
    )

    if n_repeat <= 1:
        return base

    parts = []
    for i in range(n_repeat):
        part = base.copy(deep=True)
        part["flow_id"] = [
            f"{fid}_r{i:04d}"
            for fid in part["flow_id"]
        ]
        parts.append(part)

    return pd.concat(parts, ignore_index=True)


def make_flow_to_trips_df(
    flow_ids: list[str] | None = None,
) -> pd.DataFrame:
    if flow_ids is None:
        flow_ids = ["f_0001", "f_0002", "f_0003"]

    rows = []
    for idx, fid in enumerate(flow_ids):
        rows.append(
            {
                "flow_id": fid,
                "movement_id": f"m_{idx * 2 + 1:04d}",
            }
        )
        rows.append(
            {
                "flow_id": fid,
                "movement_id": f"m_{idx * 2 + 2:04d}",
            }
        )

    return pd.DataFrame(rows)


def make_flowdataset(
    *,
    validated: bool = True,
    with_flow_to_trips: bool = False,
    dataset_id: str = "flow-dset-smoke-001",
) -> FlowDataset:
    flows_df = make_flow_df()

    flow_to_trips_df = (
        make_flow_to_trips_df(flows_df["flow_id"].tolist())
        if with_flow_to_trips
        else None
    )

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "min_trips_per_flow": 1,
    }

    metadata = {
        "dataset_id": dataset_id,
        "is_validated": validated,
        "events": [],
        "notes": {"smoke_case": True},
    }

    provenance = {
        "derived_from": [
            {
                "source_type": "trips",
                "dataset_id": "trip-dset-origin-001",
            }
        ],
        "prior_events_summary": {"n_events": 2},
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=aggregation_spec,
        source_trips={"debug_only": True},
        metadata=metadata,
        provenance=provenance,
    )


root = reset_smoke_root()
print("SMOKE_ROOT =", root.resolve())
show_ok("Bloque 1 - setup visible de smoke tests de OP-10 write_flows")

SMOKE_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_smoke_write_flows
OK - Bloque 1 - setup visible de smoke tests de OP-10 write_flows


## Bloque 2. Smoke test happy path mínimo de `write_flows`

Qué prueba:

- escritura formal exitosa de un artefacto de flows sin auxiliar `flow_to_trips`;
- uso del backend por defecto actual de OP-10: **Feather**;
- normalización automática del directorio con sufijo `.golondrina`;
- creación de sidecar formal;
- consistencia mínima de `summary` y `parameters`;
- actualización controlada de metadata en memoria;
- no mutación de `flows.flows`.

Este test reemplaza el antiguo happy path mínimo basado explícitamente en Parquet, porque la implementación vigente dejó **Feather** como backend de escritura por defecto. 

In [2]:
case_dir = make_case_dir("case_01_write_happy_default_feather")
artifact_dir = case_dir / "artifact_write_happy"
true_artifact_dir = case_dir / "artifact_write_happy.golondrina"

flows = make_flowdataset(
    validated=True,
    with_flow_to_trips=False,
)

flows_before = flows.flows.copy(deep=True)
metadata_before = copy.deepcopy(flows.metadata)

report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

assert report.ok is True

# Layout formal persistido
assert true_artifact_dir.exists()
assert (true_artifact_dir / "flows.feather").exists()
assert (true_artifact_dir / "flows.metadata.json").exists()
assert not (true_artifact_dir / "flow_to_trips.feather").exists()

# Summary mínimo
assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert report.summary["dataset_id"] == flows.metadata["dataset_id"]
assert report.summary["artifact_id"] == flows.metadata["artifact_id"]
assert "flows.feather" in report.summary["files_written"]
assert "flows.metadata.json" in report.summary["files_written"]

# Parameters efectivos
assert report.parameters["path"] == str(true_artifact_dir)
assert report.parameters["storage_format"] == "feather"
assert report.parameters["mode"] == "error_if_exists"
assert report.parameters["normalize_artifact_dir"] is True
assert report.parameters["write_flow_to_trips"] is False

# Side effects mínimos en metadata
assert flows.metadata["dataset_id"] == metadata_before["dataset_id"]
assert "artifact_id" in flows.metadata
assert flows.metadata["is_validated"] is True
assert flows.metadata["events"][-1]["op"] == "write_flows"

# flows.flows no debe mutar
pd.testing.assert_frame_equal(
    flows.flows,
    flows_before,
)

# Sidecar consistente
sidecar = read_json(true_artifact_dir / "flows.metadata.json")

assert sidecar["dataset_type"] == "flows"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"

assert sidecar["storage"]["format"] == "feather"
assert sidecar["files"]["data"] == "flows.feather"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None

assert sidecar["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["artifact_id"] == flows.metadata["artifact_id"]

assert sidecar["metadata"]["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["metadata"]["artifact_id"] == flows.metadata["artifact_id"]
assert sidecar["metadata"]["events"][-1]["op"] == "write_flows"

display(report)
show_ok("Bloque 2 - write_flows happy path mínimo con Feather por defecto")

OperationReport(ok=True, issues=[], summary={'n_flows': 3, 'n_flow_to_trips': None, 'files_written': ['flows.feather', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_7004cbe4-469b-4946-818d-c54ae72294fd', 'path': 'tmp_smoke_write_flows\\case_01_write_happy_default_feather\\artifact_write_happy.golondrina'}, parameters={'path': 'tmp_smoke_write_flows\\case_01_write_happy_default_feather\\artifact_write_happy.golondrina', 'mode': 'error_if_exists', 'storage_format': 'feather', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': True, 'write_flow_to_trips': False})

OK - Bloque 2 - write_flows happy path mínimo con Feather por defecto


## Bloque 3. Smoke test de categóricos persistidos eficientemente en Parquet

Qué prueba:

- que `write_flows` sigue soportando explícitamente el backend Parquet;
- que un campo de segmentación incluido en `group_by`, como `mode`, queda escrito con dictionary encoding observable en Parquet;
- que el artefacto generado existe y es legible a nivel físico;
- que se conserva la comparación visible de tamaño contra una escritura manual sin dictionary encoding.

Nota:

La preparación de categóricos en la implementación actual es compartida para Parquet y Feather. Sin embargo, este smoke test conserva la comprobación observable del notebook original sobre Parquet, porque permite inspeccionar directamente el dictionary encoding del archivo persistido. 

In [3]:
import pyarrow as pa
import pyarrow.parquet as pq

case_dir = make_case_dir("case_02_categorical_encoding_parquet")
artifact_dir = case_dir / "artifact_good"
artifact_bad_dir = case_dir / "artifact_bad_manual"
artifact_bad_dir.mkdir(parents=True, exist_ok=True)

flows_big = make_flowdataset(
    validated=True,
    with_flow_to_trips=False,
)
flows_big.flows = make_flow_df(n_repeat=3000)

report = write_flows(
    flows_big,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="overwrite",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

assert report.ok is True

good_parquet = artifact_dir / "flows.parquet"
assert good_parquet.exists()

# 1) Verificación observable: dictionary encoding en "mode"
parquet_file = pq.ParquetFile(good_parquet)

try:
    names = parquet_file.schema_arrow.names
    mode_idx = names.index("mode")

    encodings = {
        str(enc).upper()
        for enc in parquet_file.metadata.row_group(0).column(mode_idx).encodings
    }

    assert any("DICTIONARY" in enc for enc in encodings), encodings
finally:
    parquet_file.close()

# 2) Escritura manual "mala" para comparar tamaño
df_bad = flows_big.flows.copy(deep=True)
table_bad = pa.Table.from_pandas(
    df_bad,
    preserve_index=False,
)

bad_parquet = artifact_bad_dir / "flows_bad_no_dictionary.parquet"

pq.write_table(
    table_bad,
    bad_parquet,
    compression="snappy",
    use_dictionary=False,
)

size_good = good_parquet.stat().st_size
size_bad = bad_parquet.stat().st_size

print("size_good =", size_good)
print("size_bad  =", size_bad)
print(
    "ratio bad/good =",
    round(size_bad / size_good, 3) if size_good else None,
)

assert size_good > 0
assert size_bad > 0

display(report)
show_ok("Bloque 3 - categóricos persistidos eficientemente en Parquet")

size_good = 58518
size_bad  = 74834
ratio bad/good = 1.279


OperationReport(ok=True, issues=[], summary={'n_flows': 9000, 'n_flow_to_trips': None, 'files_written': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_3b2ca48e-400a-43d4-bb70-2b529e94ef68', 'path': 'tmp_smoke_write_flows\\case_02_categorical_encoding_parquet\\artifact_good'}, parameters={'path': 'tmp_smoke_write_flows\\case_02_categorical_encoding_parquet\\artifact_good', 'mode': 'overwrite', 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': False, 'write_flow_to_trips': False})

OK - Bloque 3 - categóricos persistidos eficientemente en Parquet


## Bloque 4. Smoke test de `write_flows` con auxiliar `flow_to_trips`

Qué prueba:

- persistencia correcta del auxiliar opcional `flow_to_trips` cuando existe en memoria y se solicita escribirlo;
- comportamiento coherente con el backend por defecto actual de OP-10: **Feather**;
- materialización de `flow_to_trips.feather`;
- declaración correcta del auxiliar en `summary` y sidecar;
- conservación de `n_flow_to_trips`.

Este bloque proviene del antiguo smoke test integrado `write/read` con auxiliar, pero fue separado para dejar únicamente la parte correspondiente a **OP-10 `write_flows`**. Además, se corrigió la expectativa antigua de `flow_to_trips.parquet`, porque hoy el backend por defecto de escritura es Feather. 

In [4]:
case_dir = make_case_dir("case_03_write_with_aux_present")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(
    validated=True,
    with_flow_to_trips=True,
)

report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

assert report.ok is True

# Layout físico esperado con backend Feather por defecto
assert artifact_dir.exists()
assert (artifact_dir / "flows.feather").exists()
assert (artifact_dir / "flows.metadata.json").exists()
assert (artifact_dir / "flow_to_trips.feather").exists()

# Summary
assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)

assert "flows.feather" in report.summary["files_written"]
assert "flows.metadata.json" in report.summary["files_written"]
assert "flow_to_trips.feather" in report.summary["files_written"]

# Parameters
assert report.parameters["storage_format"] == "feather"
assert report.parameters["write_flow_to_trips"] is True
assert report.parameters["normalize_artifact_dir"] is False

# Sidecar
sidecar = read_json(artifact_dir / "flows.metadata.json")

assert sidecar["storage"]["format"] == "feather"
assert sidecar["files"]["data"] == "flows.feather"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] == "flow_to_trips.feather"

assert sidecar["tables"]["flow_to_trips"] is not None
assert sidecar["tables"]["flow_to_trips"]["n_rows"] == len(flows.flow_to_trips)

# Metadata final alineada con el write
assert flows.metadata["events"][-1]["op"] == "write_flows"
assert flows.metadata["artifact_id"] == report.summary["artifact_id"]

display(report)
show_ok("Bloque 4 - write_flows con flow_to_trips existente")

OperationReport(ok=True, issues=[], summary={'n_flows': 3, 'n_flow_to_trips': 6, 'files_written': ['flows.feather', 'flows.metadata.json', 'flow_to_trips.feather'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_dd51b91a-4ecc-4329-b774-7d020baf3504', 'path': 'tmp_smoke_write_flows\\case_03_write_with_aux_present\\artifact'}, parameters={'path': 'tmp_smoke_write_flows\\case_03_write_with_aux_present\\artifact', 'mode': 'error_if_exists', 'storage_format': 'feather', 'parquet_compression': 'snappy', 'feather_compression': 'lz4', 'normalize_artifact_dir': False, 'write_flow_to_trips': True})

OK - Bloque 4 - write_flows con flow_to_trips existente
